In [1]:
import pandas as pd
import numpy as np

test = pd.read_csv("test.csv")
test.head()
test.shape

(1459, 80)

In [2]:
train= pd.read_csv("train.csv")
train.head()
train.shape

(1460, 81)

In [3]:
## concat both and fill NaN 

In [4]:
test["SalePrice"] = -1
data = pd.concat([train, test], ignore_index=True)

In [5]:
mode_cols = ["SaleType", "Exterior1st", "Exterior2nd", "Electrical", "KitchenQual", "Functional", "Utilities", "MSZoning"]

none_cols = ["PoolQC", "Alley", "GarageType", "GarageFinish", "GarageQual", "GarageCond","MiscFeature", "Fence", "FireplaceQu", "BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2"]

In [6]:
data[none_cols] = data[none_cols].fillna("None")
data[mode_cols] = data[mode_cols].fillna(data[mode_cols].mode().iloc[0])

In [7]:
data.loc[data["TotalBsmtSF"] == 0,
         ["BsmtFinSF1", "BsmtFinSF2", "BsmtUnfSF",
          "BsmtFullBath", "BsmtHalfBath"]] = 0

In [8]:
data.loc[data["BsmtQual"] == "None" ,
         ["BsmtFinSF1","TotalBsmtSF", "BsmtFinSF2", "BsmtUnfSF",
          "BsmtFullBath", "BsmtHalfBath"]] = 0

In [9]:
data.loc[data["GarageType"] == "None",
         ["GarageArea","GarageCars"]] = 0

In [10]:
((data["MasVnrType"].isnull()) & (data["MasVnrArea"].isnull())).sum()

np.int64(23)

In [11]:
mask = data["MasVnrArea"].isnull() | (data["MasVnrArea"] == 0)
data.loc[mask, "MasVnrArea"] = 0
data.loc[mask, "MasVnrType"] = "None"

In [12]:
data.loc[
    data["MasVnrType"].isnull() & (data["MasVnrArea"] > 0),
    "MasVnrType"
] = data["MasVnrType"].mode()[0]

In [13]:
((data["GarageType"] == "None") & (data["GarageArea"].isnull())).sum()

np.int64(0)

In [14]:
data[data["GarageCars"].isnull()][["GarageType", "GarageArea", "GarageCars"]]

,GarageType,GarageArea,GarageCars
2576,Detchd,NaN,NaN


In [15]:
data["GarageCars"] = data["GarageCars"].fillna(data["GarageCars"].median())
data["GarageArea"] = data["GarageArea"].fillna(data["GarageArea"].median())

In [16]:
data["LotFrontage"] = data["LotFrontage"].fillna(data["LotFrontage"].median())

In [17]:
data.loc[data["GarageType"] == "None", "GarageYrBlt"] = 0

In [18]:
data["GarageYrBlt"] = data["GarageYrBlt"].fillna(data["GarageYrBlt"].median())

In [19]:
missing = data.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print(missing)
## missing evaluated

Series([], dtype: int64)


In [20]:
data["TotalSF"] = data["TotalBsmtSF"] + data["1stFlrSF"] + data["2ndFlrSF"]

data["TotalBath"] = data["FullBath"] + 0.5*data["HalfBath"] + data["BsmtFullBath"] + 0.5*data["BsmtHalfBath"]

data["HouseAge"] = data["YrSold"] - data["YearBuilt"]

data["RemodAge"] = data["YrSold"] - data["YearRemodAdd"]

data["TotalPorchSF"] = data["OpenPorchSF"] + data["EnclosedPorch"] + data["3SsnPorch"] + data["ScreenPorch"]

In [21]:
## Handling categorial features

In [22]:
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

In [23]:
ordinal_cols = ["LotShape", "Utilities", "LandSlope", "ExterQual", "ExterCond", "BsmtQual","Fence", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2", "HeatingQC", "KitchenQual", "Functional", "FireplaceQu", "GarageFinish", "GarageQual", "GarageCond", "PavedDrive", "PoolQC"]

nominal_cols = ["MSZoning", "Street", "Alley", "LandContour", "LotConfig", "Neighborhood","MiscFeature", "Condition1", "Condition2", "BldgType", "HouseStyle", "RoofStyle", "RoofMatl", "Exterior1st", "Exterior2nd", "MasVnrType", "Foundation", "Heating", "CentralAir", "Electrical", "GarageType", "SaleType", "SaleCondition"]

In [24]:
ordinal_encoder = OrdinalEncoder()
ohe_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
data[ordinal_cols] = ordinal_encoder.fit_transform(data[ordinal_cols])

encoded = ohe_encoder.fit_transform(data[nominal_cols])
encoded = pd.DataFrame(
    encoded,
    columns=ohe_encoder.get_feature_names_out(nominal_cols),
    index=data.index
)

data = data.drop(columns=nominal_cols)
data = pd.concat([data, encoded], axis=1)

In [25]:
data = data.drop(data[(data["GrLivArea"] > 4000) & (data["SalePrice"] < 300000) & (data["SalePrice"] >-1)].index)

In [26]:
data[(data["GarageArea"] > 1200) & (data["SalePrice"] < 200000)& (data["SalePrice"] >-1)]

,Id,MSSubClass,LotFrontage,LotArea,LotShape,Utilities,LandSlope,OverallQual,OverallCond,YearBuilt,...,SaleType_ConLw,SaleType_New,SaleType_Oth,SaleType_WD,SaleCondition_Abnorml,SaleCondition_AdjLand,SaleCondition_Alloca,SaleCondition_Family,SaleCondition_Normal,SaleCondition_Partial
1061,1062,30,120.0,18000,3.0,0.0,0.0,3,4,1935,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1190,1191,190,68.0,32463,3.0,0.0,1.0,4,4,1961,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0


In [27]:
data = data.drop(data[(data["GarageArea"] > 1200) & (data["SalePrice"] < 200000)&(data["SalePrice"] >-1)].index)

In [28]:
data[(data["TotalSF"] > 5000) & (data["SalePrice"] < 300000)&(data["SalePrice"] >-1)]

,Id,MSSubClass,LotFrontage,LotArea,LotShape,Utilities,LandSlope,OverallQual,OverallCond,YearBuilt,...,SaleType_ConLw,SaleType_New,SaleType_Oth,SaleType_WD,SaleCondition_Abnorml,SaleCondition_AdjLand,SaleCondition_Alloca,SaleCondition_Family,SaleCondition_Normal,SaleCondition_Partial
1044,1045,20,80.0,9600,3.0,0.0,0.0,8,5,1981,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0


In [29]:
data = data.drop(data[(data["TotalSF"] > 5000) & (data["SalePrice"] < 100000)&(data["SalePrice"] >-1)].index)

In [30]:
data[(data["OverallQual"] >= 9) & (data["SalePrice"] < 300000)&(data["SalePrice"] >-1)]

,Id,MSSubClass,LotFrontage,LotArea,LotShape,Utilities,LandSlope,OverallQual,OverallCond,YearBuilt,...,SaleType_ConLw,SaleType_New,SaleType_Oth,SaleType_WD,SaleCondition_Abnorml,SaleCondition_AdjLand,SaleCondition_Alloca,SaleCondition_Family,SaleCondition_Normal,SaleCondition_Partial
34,35,120,60.0,7313,3.0,0.0,0.0,9,5,2005,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
343,344,120,63.0,8849,0.0,0.0,0.0,9,5,2005,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
683,684,20,90.0,11248,0.0,0.0,0.0,9,5,2002,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
765,766,20,75.0,14587,0.0,0.0,0.0,9,5,2008,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
963,964,20,122.0,11923,0.0,0.0,0.0,9,5,2007,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0


In [31]:
data = data.drop(data[(data["OverallQual"] >= 9) & (data["SalePrice"] < 300000)&(data["SalePrice"] >-1)].index)

In [32]:
data.shape

(2910, 232)

In [33]:
df1 = data[data["SalePrice"] != -1].copy()
df2 = data[data["SalePrice"] == -1].drop(columns="SalePrice").copy()

X = df1.drop(columns="SalePrice")
y = np.log1p(df1["SalePrice"])

from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split( X, y, test_size=0.2, random_state=42)
## Not my idea but to calculate accuracy this is best

In [34]:
X_train.shape

(1160, 231)

In [35]:
df2.shape

(1459, 231)

In [36]:
df1.shape

(1451, 232)

In [37]:
## numerical vals scaling

In [38]:
num_cols = X_train.select_dtypes(include=["int64", "float64"]).columns

In [39]:
print(num_cols)

Index(['Id', 'MSSubClass', 'LotFrontage', 'LotArea', 'LotShape', 'Utilities',
       'LandSlope', 'OverallQual', 'OverallCond', 'YearBuilt',
       ...
       'SaleType_ConLw', 'SaleType_New', 'SaleType_Oth', 'SaleType_WD',
       'SaleCondition_Abnorml', 'SaleCondition_AdjLand',
       'SaleCondition_Alloca', 'SaleCondition_Family', 'SaleCondition_Normal',
       'SaleCondition_Partial'],
      dtype='object', length=231)


In [40]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_val[num_cols] = scaler.transform(X_val[num_cols])

df2[num_cols] = scaler.transform(df2[num_cols])

In [41]:
pip install xgboost

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [42]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import cross_val_score

In [43]:
xgb = XGBRegressor(
    n_estimators=700,
    learning_rate=0.1,
    max_depth=3,
    subsample=0.8,
    colsample_bytree=0.6,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

xgb.fit(X_train, y_train)
y_pred = np.expm1(xgb.predict(X_val))
y_true = np.expm1(y_val)

print("RMSE:", np.sqrt(mean_squared_error(y_true, y_pred)))
print("MAE:", mean_absolute_error(y_true, y_pred))
print("R2:", r2_score(y_true, y_pred))

RMSE: 23817.47799141985
MAE: 14059.531129188144
R2: 0.9175753567773155


In [45]:
test_pred = np.expm1(xgb.predict(df2))
submission = pd.DataFrame({
    "Id": test["Id"],
    "SalePrice": test_pred
})

In [46]:
submission.to_csv("submission5_outliers.csv", index=False)

In [47]:
print(submission.head())
print(submission.columns)

     Id      SalePrice
0  1461  121998.820312
1  1462  169352.484375
2  1463  190557.562500
3  1464  198978.359375
4  1465  174857.859375
Index(['Id', 'SalePrice'], dtype='object')


In [48]:
submission.shape
submission.columns

Index(['Id', 'SalePrice'], dtype='object')